In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.cluster import AgglomerativeClustering
from scipy.stats import chi2_contingency

In [2]:
expression = pd.read_csv("shared_expression_log2.csv", index_col=0)
sample_meta = pd.read_csv("shared_sample_metadata.csv", index_col=0)
print(expression.shape)                                   # (32, 32309)
print((expression.index == sample_meta.index).all())      # True

(32, 32309)
True


In [3]:
gene_variance = expression.var(axis=0)
expression_5000 = expression.loc[:, gene_variance.nlargest(5000).index]


In [4]:
k_values = [2, 3, 4, 5, 6]
for k in k_values:
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    sample_meta[f"hier_k{k}"] = model.fit_predict(expression_5000)

k_columns = [f"hier_k{k}" for k in k_values]

In [5]:
for a, b in zip(k_columns[:-1], k_columns[1:]):
    print(f"\n{a} vs {b}")
    print(pd.crosstab(sample_meta[a], sample_meta[b]))


hier_k2 vs hier_k3
hier_k3  0   1  2
hier_k2          
0        9   0  8
1        0  15  0

hier_k3 vs hier_k4
hier_k4  0   1  2  3
hier_k3             
0        8   0  0  1
1        0  15  0  0
2        0   0  8  0

hier_k4 vs hier_k5
hier_k5   0  1  2  3  4
hier_k4                
0         0  6  0  0  2
1        15  0  0  0  0
2         0  0  8  0  0
3         0  0  0  1  0

hier_k5 vs hier_k6
hier_k6  0  1  2  3  4  5
hier_k5                  
0        0  7  8  0  0  0
1        6  0  0  0  0  0
2        0  0  0  0  0  8
3        0  0  0  1  0  0
4        0  0  0  0  2  0


In [6]:
gene_counts = [10, 100, 1000, 5000, 10000]
for n in gene_counts:
    subset = expression.loc[:, gene_variance.nlargest(n).index]
    sample_meta[f"hier_{n}_genes"] = AgglomerativeClustering(
        n_clusters=2, linkage="ward").fit_predict(subset)

gene_cols = [f"hier_{n}_genes" for n in gene_counts]

In [7]:
rows = []
for c1, c2 in combinations(gene_cols, 2):
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(sample_meta[c1], sample_meta[c2]))
    rows.append({"Clustering_1": c1, "Clustering_2": c2,
                 "Chi_Square": chi2, "DoF": dof, "P_Value": p})
gene_chi = pd.DataFrame(rows)
gene_chi.to_csv("results/hier_gene_count_chi_square_raw.csv", index=False)


In [8]:
rows = []
for col in k_columns + gene_cols:
    for group in ["condition", "genotype", "age"]:
        chi2, p, dof, _ = chi2_contingency(pd.crosstab(sample_meta[col], sample_meta[group]))
        rows.append({"Clustering": col, "Compared_To": group,
                     "Chi_Square": chi2, "DoF": dof, "P_Value": p})
group_chi = pd.DataFrame(rows)
group_chi.to_csv("results/hier_chi_square_vs_groups_raw.csv", index=False)
group_chi

,Clustering,Compared_To,Chi_Square,DoF,P_Value
0,hier_k2,condition,0.000000,1,1.000000e+00
1,hier_k2,genotype,24.596078,1,7.069546e-07
2,hier_k2,age,0.000000,1,1.000000e+00
3,hier_k3,condition,0.177778,2,9.149472e-01
4,hier_k3,genotype,28.444444,2,6.658361e-07
5,hier_k3,age,13.511111,2,1.164393e-03
6,hier_k4,condition,1.066667,3,7.851264e-01
7,hier_k4,genotype,32.000000,3,5.233466e-07
8,hier_k4,age,17.066667,3,6.847767e-04
9,hier_k5,condition,3.733333,4,4.432964e-01


In [9]:
sample_meta.to_csv("results/hier_cluster_assignments.csv")

In [10]:
print(pd.crosstab(sample_meta["genotype"], sample_meta["hier_k2"]))
print(pd.crosstab([sample_meta["genotype"], sample_meta["age"]], sample_meta["hier_k4"]))
print("Singleton at k=4:", sample_meta.index[sample_meta["hier_k4"] == 3].tolist())

hier_k2    0   1
genotype        
Col-0      1  15
WS        16   0
hier_k4       0  1  2  3
genotype age            
Col-0    4d   0  8  0  0
         8d   0  7  0  1
WS       4d   8  0  0  0
         8d   0  0  8  0
Singleton at k=4: ['Spaceflight, 8 days old, Col-0 Rep2']
